In [1]:
!pip install git+https://github.com/huggingface/transformers torch accelerate bitsandbytes langchain
!pip install evaluate
!PIP install datasets
!pip install git+https://github.com/google-research/bleurt.git
!pip install langchain_community
!pip install peft
!pip install rouge_score
!pip install bert_score

  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-dqkalo11
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-dqkalo11
  Resolved https://github.com/huggingface/transformers to commit 85817d98fb60977c97e3014196a462b732d2ed1a
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl.

In [3]:
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
from langchain.llms import HuggingFacePipeline
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [4]:
from langchain.docstore.document import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from peft import PeftModel
from langchain.retrievers import EnsembleRetriever

In [6]:
import pandas as pd
test1 = pd.read_csv("/content/claude.csv")
test2 = pd.read_csv("/content/gptt.csv")

In [9]:
test1.columns

Index(['Responses'], dtype='object')

In [12]:
predictions1 = []
for p in test1["Responses"]:
  predictions1.append(p)

In [14]:
predictions2 = []
for p in test2["responses"]:
  predictions2.append(p)

In [23]:
references = []

df = pd.read_csv("/content/testt.csv")

for r in df["response"]:
  references.append(r)

references = references[:50]

In [19]:
from bert_score import score
from evaluate import load
rouge = load('rouge')
bertscore = load('bertscore')
bleu = load('bleu')

In [25]:
predictions_fixed = [p for p in predictions1]
references_fixed = [r for r in references]

rouge_scores = rouge.compute(predictions=predictions_fixed, references=references_fixed)
bleu_score = bleu.compute(predictions=predictions_fixed, references=references_fixed)

P, R, F1 = score(predictions_fixed, references_fixed, lang="en", verbose=True)

bert_scores = {"precision": P.mean().item(), "recall": R.mean().item(), "f1": F1.mean().item()}

print("ROUGE scores:", rouge_scores)
print("BERT scores:", bert_scores)
print("BLEU score:", bleu_score)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/2 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 3.56 seconds, 14.03 sentences/sec
ROUGE scores: {'rouge1': 0.17536417009240546, 'rouge2': 0.021947090909673693, 'rougeL': 0.10649358460846411, 'rougeLsum': 0.10688310066783693}
BERT scores: {'precision': 0.8634088635444641, 'recall': 0.8239672780036926, 'f1': 0.8431380987167358}
BLEU score: {'bleu': 0.004246390524353988, 'precisions': [0.3114468332364904, 0.019748653500897665, 0.00431832202344232, 0.0006365372374283895], 'brevity_penalty': 0.3723958818737977, 'length_ratio': 0.5030692779888921, 'translation_length': 1721, 'reference_length': 3421}


In [26]:
predictions_fixed = [p for p in predictions2]
references_fixed = [r for r in references]

rouge_scores = rouge.compute(predictions=predictions_fixed, references=references_fixed)
bleu_score = bleu.compute(predictions=predictions_fixed, references=references_fixed)

P, R, F1 = score(predictions_fixed, references_fixed, lang="en", verbose=True)

bert_scores = {"precision": P.mean().item(), "recall": R.mean().item(), "f1": F1.mean().item()}

print("ROUGE scores:", rouge_scores)
print("BERT scores:", bert_scores)
print("BLEU score:", bleu_score)

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/2 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 1.93 seconds, 25.90 sentences/sec
ROUGE scores: {'rouge1': 0.19677150305187824, 'rouge2': 0.022598108921472737, 'rougeL': 0.11965578697152467, 'rougeLsum': 0.11995570493389032}
BERT scores: {'precision': 0.852130651473999, 'recall': 0.824160635471344, 'f1': 0.8378342986106873}
BLEU score: {'bleu': 0.004467761650342761, 'precisions': [0.28681375064135456, 0.018957345971563982, 0.0027041644131963224, 0.0005558643690939411], 'brevity_penalty': 0.46988884735606434, 'length_ratio': 0.5697164571762643, 'translation_length': 1949, 'reference_length': 3421}
